# 13 · From pixels to data – invoices, receipts and Content Understanding

OCR gives text. Business needs **fields**: vendor, invoice number, total, line items – with a **confidence** per value so you know what to double-check.

| | Use | Engine |
|---|---|---|
| **Document Intelligence prebuilt models** (`prebuilt-invoice`, `prebuilt-receipt`, ...) | common, structured document types; fast, deterministic, proven | specialised vision models |
| **Content Understanding** | your own schema, varied or complex documents, inferred fields, summaries, classification | LLM + OCR + grounding, needs model deployments |

Then we make it an **agent tool**: the agent decides *when* to read a document, the service does the reading. Deep dive: [docs/foundry-tools.md](../docs/foundry-tools.md).

In [ ]:
import sys; sys.path.insert(0, '.')
from stark_helpers import *
from stark_docs import *

import pandas as pd
from IPython.display import Markdown, display

## 1 · `prebuilt-invoice`
`extract_invoice()` ([stark_docs.py](stark_docs.py)) runs `prebuilt-invoice` and returns plain Python: `fields`, a `confidence` per field and the line `items`. The next chapters reuse it as a tool.

**Expected:** the values printed on [invoice-pym-4471.pdf](../data/docs/invoice-pym-4471.pdf) (see [make_sample_docs.py](../scripts/make_sample_docs.py)):

| Field | Value |
|---|---|
| `VendorName` | Pym Particle Supply Co. |
| `InvoiceId` | 4471 |
| `InvoiceDate` / `DueDate` | 2026-10-02 / 2026-11-30 |
| `PurchaseOrder` | PO-7781 |
| `SubTotal` / `TotalTax` / `InvoiceTotal` | 3085.5 / 246.84 / 3332.34 |
| `CustomerName` | Stark Industries |

The second table lists three line items: Pym particles (4 × 450.00), the shrink/grow regulator (1,200.00) and shipping (85.50).

In [ ]:
pym_invoice = extract_invoice("data/docs/invoice-pym-4471.pdf")
pd.DataFrame({"value": pym_invoice["fields"], "confidence": pd.Series(pym_invoice["confidence"])})

In [ ]:
pd.DataFrame(pym_invoice["items"])

### Trust, but verify
Confidence below a threshold goes to a human; and arithmetic is checked **in code**, not by the model. Both steps use `pym_invoice` from section 1.

### Step 1 · Low confidence goes to a human

**Goal:** list the fields whose confidence is below `CONFIDENCE_THRESHOLD` (0.8). Those are the ones a person double-checks.

**Expected:** ❓ depends on the run. In one test run (chapter 15) Document Intelligence read `InvoiceTotal` with confidence 0.44, so it would be listed here. A field it did not find has no confidence and is not listed.

In [ ]:
CONFIDENCE_THRESHOLD = 0.8
low_confidence = {field: confidence for field, confidence in pym_invoice["confidence"].items() if confidence < CONFIDENCE_THRESHOLD}
print("needs human review:", low_confidence or "nothing")

### Step 2 · Arithmetic in code

**Goal:** check in Python that the line items add up to the subtotal. A rule like this belongs in code: it gives the same answer every time.

**Expected:** `line items 3,085.50 vs subtotal 3,085.50 -> OK` (1,800.00 + 1,200.00 + 85.50). `MISMATCH` means an amount was misread, which is what this check is for.

In [ ]:
line_items_total = sum(float(item["amount"] or 0) for item in pym_invoice["items"])
subtotal = float(pym_invoice["fields"]["SubTotal"] or 0)
print(f"line items {line_items_total:,.2f} vs subtotal {subtotal:,.2f} ->", "OK" if abs(line_items_total - subtotal) < 0.01 else "MISMATCH")

## 2 · `prebuilt-receipt`
Same call, another model id. `di_analyze()` returns the raw result; `field_value()` (both in [stark_docs.py](stark_docs.py)) picks the typed value of a field.

**Expected:** `MerchantName` DUM-E'S CAFE, `TransactionDate` 2026-10-11, `TotalTax` 1.08, `Total` 14.58. ❓ The receipt prints no subtotal line, so `Subtotal` may be `None`.

In [ ]:
receipt_result = di_analyze("prebuilt-receipt", "data/docs/receipt-dummy-cafe.png")
receipt = receipt_result.documents[0]
for field_name in ("MerchantName", "TransactionDate", "Subtotal", "TotalTax", "Total"):
    field = receipt.fields.get(field_name)
    print(f"{field_name:16} {field_value(field)}   (conf {field.get('confidence') if field else '-'})")

## 3 · Content Understanding
An **analyzer** defines what to extract. Prebuilt analyzers exist (`prebuilt-invoice`, `prebuilt-documentSearch` for Markdown + figures). CU uses **LLM deployments in your Foundry resource**, so they must exist and be registered as *defaults*:

```bash
azd env set STARK_ENABLE_CU true && azd provision      # deploys gpt-4.1, gpt-4.1-mini, text-embedding-3-large
```

Without them, `CU_ENABLED` is `False` and the Content Understanding cells skip.

In [ ]:
CU_ENABLED = cfg("STARK_ENABLE_CU", "false").lower() == "true"
print("Content Understanding models deployed:", CU_ENABLED)
if CU_ENABLED:
    # one-time: tell CU which of your deployments to use for each model family
    cu_client().update_defaults(model_deployments={
        "gpt-4.1": "gpt-4-1", "gpt-4.1-mini": "gpt-4-1-mini", "text-embedding-3-large": "text-embedding-3-large"})

### The prebuilt invoice analyzer

The Oscorp invoice through CU's `prebuilt-invoice`. `cu_fields()` ([stark_docs.py](stark_docs.py)) flattens the result to field, value and confidence.

**Expected:** ❓ not checked live. The vendor Oscorp Materials and the total 99,792.00 (92,400.00 + 8 % tax) should be among the fields; CU's field names can differ from Document Intelligence's.

In [ ]:
if CU_ENABLED:
    prebuilt_cu_result = cu_analyze("prebuilt-invoice", "data/docs/invoice-oscorp-9930.pdf")
    display(pd.DataFrame(cu_fields(prebuilt_cu_result)))

### A custom analyzer with your own schema
Three field methods: **extract** (literal text), **generate** (the model writes it), **classify** (pick from a list). `estimate_source_and_confidence` adds grounding. The cell creates the analyzer, runs it on the Oscorp invoice and deletes it again, even when the analysis fails.

**Expected:** ❓ not checked live. `vendor` Oscorp Materials and `total` 99792 with a confidence; `category` most likely `materials` (vibranium sheets plus freight); `risk_note` is a sentence the model writes, different on every run.

In [ ]:
import time
if CU_ENABLED:
    from azure.ai.contentunderstanding.models import (ContentAnalyzer, ContentAnalyzerConfig, ContentFieldSchema,
                                                      ContentFieldDefinition, ContentFieldType, GenerationMethod)
    analyzer_id = f"stark_invoice_{int(time.time())}"
    schema = ContentFieldSchema(name="stark_invoice", description="Fields the workshop cares about", fields={
        "vendor": ContentFieldDefinition(type=ContentFieldType.STRING, method=GenerationMethod.EXTRACT,
                                         description="Name of the selling company", estimate_source_and_confidence=True),
        "total": ContentFieldDefinition(type=ContentFieldType.NUMBER, method=GenerationMethod.EXTRACT,
                                        description="Total amount due in USD", estimate_source_and_confidence=True),
        "risk_note": ContentFieldDefinition(type=ContentFieldType.STRING, method=GenerationMethod.GENERATE,
                                            description="One sentence: anything unusual a finance reviewer should check"),
        "category": ContentFieldDefinition(type=ContentFieldType.STRING, method=GenerationMethod.CLASSIFY,
                                           description="Spend category", enum=["materials", "freight", "services", "other"]),
    })
    cu_client().begin_create_analyzer(analyzer_id=analyzer_id, resource=ContentAnalyzer(
        base_analyzer_id="prebuilt-document", description="Stark invoice triage",
        config=ContentAnalyzerConfig(enable_layout=True, enable_ocr=True, return_details=True),
        field_schema=schema,
        models={"completion": "gpt-4.1", "embedding": "text-embedding-3-large"},
    )).result()
    try:
        custom_cu_result = cu_analyze(analyzer_id, "data/docs/invoice-oscorp-9930.pdf")
        display(pd.DataFrame(cu_fields(custom_cu_result)))
    finally:
        cu_client().delete_analyzer(analyzer_id=analyzer_id)
        print("analyzer deleted")

## 4 · Make it an agent tool
The agent should not read PDFs in its head. Give it **two functions**: `extract_invoice` (Document Intelligence) and `check_approval_limit` (business rule). It decides when to call them. Both run in this notebook, from [stark_docs.py](stark_docs.py): `tool_functions` maps the name the model uses to the Python function, and `run_functions(...)` runs the function-call loop of chapter 05.

In [ ]:
from azure.ai.projects.models import PromptAgentDefinition, FunctionTool

tool_functions = {"extract_invoice": extract_invoice, "check_approval_limit": check_approval_limit}   # name the model uses -> Python function
invoice_tools = [
    FunctionTool(name="extract_invoice", description="Read an invoice file (path relative to the repo, e.g. data/docs/invoice-pym-4471.pdf) and return its fields, line items and confidences.",
                 parameters={"type": "object", "properties": {"path": {"type": "string"}}, "required": ["path"], "additionalProperties": False}, strict=True),
    FunctionTool(name="check_approval_limit", description="Check whether an amount in USD is within an approver's limit.",
                 parameters={"type": "object", "properties": {"amount": {"type": "number"}, "approver": {"type": "string"}},
                             "required": ["amount", "approver"], "additionalProperties": False}, strict=True),
]
FRIDAY_INSTRUCTIONS = "You are F.R.I.D.A.Y., the finance assistant. Use the tools for every fact. Answer in at most four sentences."
friday_agent = project().agents.create_version(agent_name="friday-docs-nb13", definition=PromptAgentDefinition(
    model=model("chat"), instructions=FRIDAY_INSTRUCTIONS, tools=invoice_tools))
print(f"created {friday_agent.name} version {friday_agent.version}")

project_openai = openai_client()
friday_conversations = []   # every conversation the next cell opens, deleted in "Clean up"

### Ask F.R.I.D.A.Y.

**Goal:** the agent decides which tools to call, and every fact in its answer comes from a tool. Each run opens a new conversation.

| Input | Value |
|---|---|
| agent | `friday-docs-nb13` (the cell above) |
| question | `approval_question`: "Read data/docs/invoice-oscorp-9930.pdf. Can Pepper Potts approve it?" |

**Expected:** two tool calls, printed as they run: `extract_invoice` with the path, then `check_approval_limit` with `amount` 99792.0 and `approver` Pepper Potts, which returns `within_limit: True` (her limit is 250,000). The answer: yes, Pepper Potts can approve the 99,792.00 invoice. ❓ The model chooses the calls and the wording; it may also mention a low-confidence field.

In [ ]:
approval_question = "Read data/docs/invoice-oscorp-9930.pdf. Can Pepper Potts approve it?"
print("Question:", approval_question)
friday_conversation = project_openai.conversations.create()
friday_conversations.append(friday_conversation.id)
friday_response = run_functions(friday_agent, approval_question, tool_functions, conversation=friday_conversation.id, oai=project_openai)
# every message of the final response, one paragraph each (output_text would glue them together)
friday_answer = "\n\n".join("".join(part.text for part in item.content if part.type == "output_text")
                             for item in friday_response.output if item.type == "message")
display(Markdown(f"**Answer**\n\n{friday_answer}"))

## 👀 In the portal
**Build → Agents → friday-docs-nb13**: the agent that calls the invoice function.

> 💡 All pages: [docs/portal-tour.md](../docs/portal-tour.md)

## 🧹 Clean up

In [ ]:
for conversation_id in friday_conversations:
    project_openai.conversations.delete(conversation_id)
delete_agent("friday-docs-nb13")

## 🎯 Recap
| Need | Use |
|---|---|
| Standard invoice/receipt/ID, speed, determinism | Document Intelligence prebuilt model |
| Your own fields, varied layouts, summaries, classification | Content Understanding custom analyzer |
| LLM answers over documents | agent + tool that calls the service |
| Always | confidence thresholds + arithmetic checks in code |

> 💡 Next: Act V – the warning. [14 · Ultron](14_ultron_guardrails_redteam.ipynb).